# 5-1. PEFT — 파라미터 효율적 튜닝 (LoRA & QLoRA)

> **📌 이 노트북에 대하여**
>
> 4-1, 4-2와 마찬가지로 **별도의 PPT 없이 이 노트북 하나로** 이론과 실습을 진행한다.
> 설명을 읽고 -> 코드를 실행하고 -> 결과를 관찰하는 순서로 따라오면 된다.

---

## 학습 목표

이 노트북을 마치면 다음을 할 수 있다.

1. **Fine-tuning과 RAG의 차이**를 설명하고, 언제 무엇을 써야 하는지 판단할 수 있다
2. Full Fine-tuning의 **메모리 요구량을 직접 계산**하고 왜 불가능한지 설명할 수 있다
3. **LoRA**가 "왜 적은 파라미터로도 작동하는지" 수식과 함께 설명할 수 있다
4. **QLoRA**가 양자화와 LoRA를 어떻게 결합하는지 이해한다
5. 16GB GPU에서 **도메인 특화 모델을 직접 학습**시키고 결과를 검증할 수 있다

---

## 목차

| # | 내용 | 성격 |
|:---:|------|:---:|
| 0 | **환경 설정** — Unsloth 설치, 모델 로딩 | 실습 |
| 1 | **왜 Fine-tuning이 필요한가** — 사전학습 모델의 한계 | 이론+실습 |
| 2 | **Full Fine-tuning의 문제** — 메모리 폭발을 직접 계산 | 이론+실습 |
| 3 | **LoRA의 원리** — 1%만 학습하는 방법 | 이론+실습 |
| 4 | **QLoRA** — 양자화 + LoRA의 결합 | 이론 |
| 5 | **실전: QLoRA 학습** — 데이터 준비부터 학습까지 | 실습 |
| 6 | **정리** | — |

---

## 선행 지식 — 앞 챕터와의 연결

| 앞 챕터 | 이번 챕터에서 어떻게 쓰이는가 |
|---|---|
| **1-2** MLP·학습 4단계 | Gradient, Optimizer, Loss 개념이 메모리 계산의 기반이 된다 |
| **3-1** 전이학습 | "가중치를 얼리고 일부만 학습"하는 발상이 그대로 확장된다 |
| **4-1** RAG | Fine-tuning과 RAG는 **경쟁 관계가 아니라 보완 관계**다 |

<br>

> **💡 이번 챕터의 한 줄 요약**
>
> 4-1(RAG): **"모델은 그대로 두고 자료를 쥐여준다"**
> 5-1(PEFT): **"모델 자체를 내 도메인에 맞게 바꾼다. 단, 1%만 건드려서."**

> **➡️ 다음 시간 예고**
>
> 이번 챕터의 **QLoRA에서 '양자화(Quantization)'** 를 사용한다.
> 여기서는 **"모델을 4-bit로 압축한다"** 는 수준으로만 다루고,
> **NF4가 무엇인지, 어떻게 압축하는지**는 **다음 시간(5-2)** 에서 자세히 배운다.


---

## 0. 환경 설정

### Unsloth란?

**Unsloth**는 LLM 파인튜닝에 최적화된 오픈소스 프레임워크이다.
표준 HuggingFace PEFT 조합 대비 **학습 속도와 메모리 효율을 크게 개선**한다.

| 항목 | 표준 HuggingFace PEFT | Unsloth |
|------|:---:|:---:|
| 학습 속도 | 기준 | **약 2배 이상 빠름** |
| 메모리 | 기준 | **수십 % 절감** |
| 코드 복잡도 | 복잡 (BitsAndBytesConfig + PeftModel 등) | **간편 (FastModel 한 줄)** |

<br>

> **⚠️ "30배 빠르다"는 표현에 대하여**
>
> Unsloth 공식 문서에 "최대 30배"라는 수치가 등장하지만, 이는
> **특정 모델·특정 설정·특정 GPU에서 측정한 최대치**다.
> 일반적인 환경에서는 **2~5배 수준**의 개선을 기대하는 것이 현실적이다.
>
> 💡 벤치마크 수치를 볼 때는 항상 **"어떤 조건에서 측정했는가"** 를 확인하는 습관을 들이자.
> 이는 논문이나 제품 홍보 자료를 읽을 때도 마찬가지다.

Unsloth는 내부적으로 **수작업 최적화된 GPU 커널(Triton)** 을 사용한다.
이 실습에서 Unsloth를 사용하는 이유는 **16GB VRAM 환경에서도 안정적으로 학습**할 수 있기 때문이다.

### Gemma란? 왜 gemma-3-1b를 사용하는가?

**Gemma**는 Google DeepMind가 공개한 오픈 웨이트(open-weight) LLM 시리즈이다.
Gemini와 기술 기반을 공유하는 경량 모델로, 연구 및 상업적 이용이 가능하다.

| 모델 | 파라미터 | 특징 |
|------|:---:|------|
| gemma-3-1b | **10억 개** | 가벼워서 16GB GPU에서 학습 가능 |
| gemma-3-4b | 40억 개 | 중간 크기. 멀티모달 지원 |
| gemma-3-12b | 120억 개 | 고성능, 고사양 GPU 필요 |
| gemma-3-27b | 270억 개 | 최고 성능, A100급 필요 |

**gemma-3-1b를 선택한 이유**:
1. 16GB VRAM에서 4-bit QLoRA 학습이 가능한 크기
2. 학습 시간이 짧아 수업 시간 내 결과 확인 가능
3. Unsloth가 미리 4-bit로 양자화한 버전을 제공
4. 대화형으로 사전학습된(`it`) 버전이라 바로 실습에 쓸 수 있음

> **📌 라이선스는 반드시 확인할 것**
>
> Gemma는 **Gemma Terms of Use**를 따른다. 완전한 MIT/Apache 라이선스가 아니며
> 사용 제한 조항이 있다. 3-1 전이학습에서 강조했듯,
> **오픈 모델을 실무에 쓰기 전에는 반드시 라이선스를 확인**해야 한다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 역할: Unsloth 및 학습에 필요한 패키지 설치 (Docker 환경이라면 실행하지 않습니다.)
#
# [각 패키지의 역할]
# - unsloth        : 최적화된 LLM 파인튜닝 프레임워크 (이 실습의 핵심)
# - peft           : LoRA 등 PEFT 기법 제공 (Unsloth가 내부적으로 사용)
# - trl            : SFTTrainer(지도 학습 미세조정 트레이너) 제공
# - datasets       : HuggingFace 데이터셋 로딩
# - accelerate     : 분산/혼합정밀도 학습 지원
# - bitsandbytes   : 4-bit 양자화 구현체 (QLoRA의 'Q'를 담당)
# - tokenizers     : 고속 토크나이저 (2-1 챕터에서 다룬 그 라이브러리)
#
# ⚠️ 설치 후 런타임(커널) 재시작이 필요할 수 있다.
#    'ModuleNotFoundError'가 뜨면 재시작 후 다시 실행하자.
# ═══════════════════════════════════════════════════════════

# %pip install unsloth
# %pip install --upgrade typing_extensions
# %pip install transformers peft trl datasets accelerate bitsandbytes tokenizers

In [ ]:
import os
# ⚠️ 반드시 unsloth import '이전'에 설정해야 한다.
#    huggingface_hub이 import 시점에 환경변수를 읽기 때문이다.
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
from unsloth import FastModel
import torch
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Unsloth로 4-bit 양자화 모델을 로딩
#
# [이 셀에서 하는 것]
# Gemma-3-1B 모델을 4-bit로 압축된 상태로 GPU에 로딩한다.
# 표준 방식으로는 BitsAndBytesConfig를 직접 설정해야 하지만,
# Unsloth는 FastModel.from_pretrained() 한 줄로 처리한다.
#
# [모델 이름 해석]
# 'unsloth/gemma-3-1b-it-unsloth-bnb-4bit'
#  ├─ unsloth/     → Unsloth가 최적화한 버전
#  ├─ gemma-3      → Google의 Gemma 3세대
#  ├─ 1b           → 10억(1 Billion) 파라미터
#  ├─ it           → Instruction-Tuned (대화형으로 사전학습됨)
#  └─ bnb-4bit     → bitsandbytes 4-bit 양자화 적용
#
# [양자화란? — 지금은 이 정도만]
# 가중치를 표현하는 비트 수를 줄여 메모리를 아끼는 기법이다.
# 원래 16비트(FP16)로 저장하던 숫자를 4비트로 줄이면 용량이 1/4이 된다.
# ➡️ 어떤 원리로 압축하는지(NF4, Double Quantization 등)는 다음 시간(5-2)에 배운다.
#    여기서는 "4-bit로 압축된 모델을 학습의 베이스로 쓴다"만 이해하면 충분하다.
#
# [확인 포인트]
# GPU 메모리가 ~1GB 수준으로 나오면 정상이다.
# 4-bit 양자화 덕분에 10억 파라미터 모델이 ~1GB에 로딩된다.
# (FP16이라면 ~2GB, FP32라면 ~4GB가 필요했을 것이다.)
#
# ⚠️ max_seq_length는 '메모리 사용량'과 직결된다.
#    이 값을 늘리면 더 긴 문맥을 학습할 수 있지만 메모리도 함께 늘어난다.
#    체스 기보는 짧으므로 1024로 충분하다.
# ═══════════════════════════════════════════════════════════

max_seq_length = 1024  # 시퀀스 최대 길이 (메모리 효율을 위해 제한)

# ★ 핵심: 4-bit 양자화 모델 로딩
model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/gemma-3-1b-it-unsloth-bnb-4bit',
    max_seq_length=max_seq_length,
    load_in_4bit=True,   # ← 4-bit 양자화로 로딩 (QLoRA의 'Q')
)

print(f'모델 로딩 완료: Gemma-3-1B (4-bit 양자화)')
if torch.cuda.is_available():
    mem = torch.cuda.memory_allocated() / 1024**3
    print(f'GPU 메모리: {mem:.2f} GB')
    print(f'→ 10억 파라미터 모델이 4-bit 양자화 덕분에 ~1GB에 로딩되었다.')

---

## 1. 왜 Fine-tuning이 필요한가?

### 1-1. 사전학습 모델의 한계

GPT, Gemma, Llama 같은 LLM은 인터넷의 방대한 텍스트로 **사전학습(Pre-training)** 되어 있다.
일반적인 질문에는 잘 답하지만, **특정 도메인이나 형식**에는 약하다.

| 요청 | 사전학습 모델 | Fine-tuning된 모델 |
|------|:---:|:---:|
| "서울 날씨 알려줘" | ✅ 잘 답함 | ✅ 잘 답함 |
| 체스 기보 → 다음 수 예측 | ❌ 형식조차 못 맞춤 | ✅ 지정한 형식으로 정확히 출력 |
| 사내 규정 기반 답변 | ❌ 사내 정보 없음 | ✅ 사내 데이터로 학습 |
| 특정 말투/형식 | ❌ 범용 말투 | ✅ 원하는 형식으로 출력 |

**Fine-tuning** = 사전학습된 모델을 **특정 작업/도메인에 맞게 추가 학습**시키는 것

> **💡 비유: 의대 졸업생과 전문의**
>
> - **사전학습 모델** = 의대를 졸업한 일반의. 기본 의학 지식은 있지만 전문 수술은 못 함
> - **Fine-tuning된 모델** = 전문의 수련을 마친 외과 전문의. 특정 분야에서 전문적 역량 발휘

### 1-2. ⭐ Fine-tuning vs RAG — 무엇을 언제 쓰는가

4-1에서 **RAG**를 배웠다. RAG도 "LLM이 모르는 것을 알게 하는" 기법이었다.
**그럼 둘은 경쟁 관계인가?** 아니다. **해결하는 문제가 다르다.**

| 구분 | RAG (4-1) | Fine-tuning (5-1) |
|------|:---:|:---:|
| 비유 | 사람에게 **자료를 쥐여준다** | 사람을 **교육시킨다** |
| 잘하는 것 | **사실 정보** 제공 | **형식·말투·행동 양식** 학습 |
| 정보가 바뀌면 | **문서만 교체** | 재학습 필요 |
| 출처 추적 | ✅ 가능 | ❌ 불가능 |
| 초기 비용 | 낮음 | **높음** (GPU + 시간) |
| 추론 비용 | 높음 (긴 프롬프트) | **낮음** (짧은 프롬프트) |

<br>

> **⭐ 판단 기준: "알아야 하는가, 할 줄 알아야 하는가"**
>
> ```
>    "우리 회사 환불 규정이 뭐야?"        -> 알아야 한다   -> RAG
>    "우리 회사 말투로 답변해줘"           -> 할 줄 알아야  -> Fine-tuning
>    "체스 기보를 보고 다음 수를 말해줘"    -> 할 줄 알아야  -> Fine-tuning  ← 이번 실습
> ```
>
> 💡 **실무에서는 둘을 함께 쓴다.**
> Fine-tuning으로 말투와 형식을 익히게 하고, RAG로 최신 사실을 공급하는 구조가 일반적이다.

<br>

> **📌 그런데 프롬프트 엔지니어링으로 안 되나?**
>
> 2-2에서 배운 **Few-shot**을 떠올려 보자. 예시 몇 개만 줘도 형식이 맞춰졌다.
> 그렇다면 Fine-tuning이 왜 필요할까?
>
> | | Few-shot 프롬프팅 | Fine-tuning |
> |---|---|---|
> | 준비 비용 | **거의 없음** | 높음 (데이터 + GPU) |
> | 예시 개수 | 몇 개 (프롬프트 길이 제한) | **수천~수만 건** |
> | 매 요청마다 | 예시를 **계속 함께 보냄** (토큰 비용) | 불필요 |
> | 복잡한 패턴 | 어렵다 | **가능** |
>
> 👉 **순서는 항상 이렇다: 프롬프팅 -> RAG -> Fine-tuning.**
> 앞의 것으로 해결되면 뒤로 갈 이유가 없다.
> 3-1에서 "LP로 먼저 해보고 아쉬우면 FT"라고 했던 것과 같은 원칙이다.

### 1-3. 이 실습의 목표

Gemma-3-1B 모델을 **체스 다음 수 예측** 전문가로 만든다.

사용할 데이터는 HuggingFace의 `Thytu/ChessInstruct`이며,
그중 **"다음 최선 수를 예측하라"** 는 과제만 골라서 사용한다.

```
   [입력]  {"moves": ["e2e4", "e7e5", "g1f3", ...]}      ← 지금까지의 수순
   [출력]  {"next best move": "d6d5"}                     ← 다음에 둘 수
```

> **💡 왜 체스인가?**
>
> 학습 효과를 **눈으로 즉시 확인**할 수 있기 때문이다.
> 학습 전에는 장황한 설명을 늘어놓던 모델이,
> 학습 후에는 **정확히 이 JSON 한 줄**로 답하게 된다.
> **"형식을 학습했다"는 것을 가장 명확하게 보여주는 예제**다.

> **⚠️ 우리가 가르치는 것은 '체스 실력'이 아니다**
>
> 5,000건 · 1에폭으로 그랜드마스터를 만들 수는 없다.
> 우리가 가르치는 것은 **"이 과제에는 이런 형식으로 답한다"** 는 행동 양식이다.
>
> ```
>    학습 목표      :  JSON 형식으로 짧게 답하기        ->  달성 가능 ✅
>    학습 목표 아님  :  최선의 수를 찾아내는 체스 실력    ->  기대하면 안 됨
> ```
>
> 👉 이 구분이 **1-2절에서 배운 RAG vs Fine-tuning의 원칙**을 그대로 실증한다.
> **Fine-tuning은 "할 줄 알게" 만들지, "알게" 만들지 않는다.**


### 1-4. 체스 기보 읽는 법 — 우리 데이터의 형식

체스를 몰라도 실습에는 지장이 없지만, **결과를 해석하려면 최소한의 규칙**은 알아야 한다.
특히 **우리 데이터가 어떤 표기법을 쓰는지**를 알아야 학습 결과를 판정할 수 있다.

#### ① 좌표 체계 — 모든 칸에 이름이 있다

체스판은 8×8이며, **가로는 알파벳(`a~h`), 세로는 숫자(`1~8`)** 로 부른다.
`e4`는 "e열 4행"이라는 뜻이다. 엑셀 셀 주소와 같은 방식이다.

```
      a    b    c    d    e    f    g    h
    ┌────┬────┬────┬────┬────┬────┬────┬────┐
  8 │    │    │    │    │    │    │    │    │ 8   ← 흑(Black) 진영
    ├────┼────┼────┼────┼────┼────┼────┼────┤
  7 │    │    │    │    │ e7 │    │    │    │ 7
    ├────┼────┼────┼────┼────┼────┼────┼────┤
  6 │    │    │    │    │    │    │    │    │ 6
    ├────┼────┼────┼────┼────┼────┼────┼────┤
  5 │    │    │    │    │ e5 │    │    │    │ 5
    ├────┼────┼────┼────┼────┼────┼────┼────┤
  4 │    │    │    │    │ e4 │    │    │    │ 4
    ├────┼────┼────┼────┼────┼────┼────┼────┤
  3 │    │    │    │    │    │    │    │    │ 3
    ├────┼────┼────┼────┼────┼────┼────┼────┤
  2 │    │    │    │    │ e2 │    │    │    │ 2
    ├────┼────┼────┼────┼────┼────┼────┼────┤
  1 │    │    │    │    │    │    │    │    │ 1   ← 백(White) 진영
    └────┴────┴────┴────┴────┴────┴────┴────┘
      a    b    c    d    e    f    g    h
```

#### ② ⭐ UCI 표기법 — 우리 데이터가 쓰는 방식

우리 학습 데이터는 **UCI(좌표) 표기법**을 쓴다. 규칙이 아주 단순하다.

> **`출발칸` + `도착칸` 을 그대로 이어 쓴다.**

```
   e2e4   →   e2 에 있는 기물을 e4 로 옮긴다
   g1f3   →   g1 에 있는 기물을 f3 으로
   d7d5   →   d7 에 있는 기물을 d5 로
```

**①에서 배운 좌표를 두 번 이어 쓴 것**이 전부다.
기물 이름도, 특수 기호도 없다. 그래서 **프로그램이 처리하기 쉽다.**

> **💡 왜 기물 이름이 필요 없을까?**
>
> 출발칸을 명시하기 때문이다. `e2`에 무엇이 있는지는 판을 보면 알 수 있으므로,
> **"e2에서 e4로"** 라고만 해도 어떤 기물이 움직였는지 확정된다.

#### ③ 실제 데이터는 이렇게 생겼다

```json
[입력 input]
{"moves": ["e2e4", "e7e5", "g1f3", "f7f5", "f1c4", "f5e4", ...]}

[정답 expected_output]
{"next best move": "d6d5"}
```

| 항목 | 형식 | 설명 |
|---|---|---|
| `input` | **JSON** | 지금까지 둔 수를 UCI로 나열한 배열 |
| `expected_output` | **JSON** | 다음에 둘 수 하나 |

<br>

> **⭐ 여기서 학습 목표가 명확해진다**
>
> 모델이 배워야 할 것은 두 가지다.
>
> ```
>    ① "체스 수순을 받으면 -> JSON 한 줄로 답한다"   ← 형식 (학습 가능 ✅)
>    ② "그 수가 최선이어야 한다"                      ← 실력 (5,000건으로는 무리)
> ```
>
> 마지막 셀에서 이 둘을 **분리해서 채점**한다.

#### ④ [참고] SAN 표기법 — 사람이 읽는 방식

체스 자료를 찾아보면 **SAN(대수기보)** 이라는 다른 표기법도 자주 만난다.
우리 데이터는 쓰지 않지만, 알아두면 자료를 읽을 때 도움이 된다.

| 기호 | 기물 | 영어 |
|:---:|---|---|
| (없음) | 폰 | Pawn |
| N | 나이트 | K**n**ight |
| B | 비숍 | **B**ishop |
| R | 룩 | **R**ook |
| Q | 퀸 | **Q**ueen |
| K | 킹 | **K**ing |

```
   [SAN]   Nf3      "나이트를 f3으로"        ← 도착칸만 적는다
   [UCI]   g1f3     "g1에서 f3으로"          ← 출발칸도 적는다
```

| | SAN | **UCI** |
|---|---|---|
| 예시 | `e4`, `Nf3`, `a6` | **`e2e4`, `g1f3`, `a7a6`** |
| 방식 | "무엇이 어디로" | **"어디서 어디로"** |
| 강점 | 사람이 읽기 쉽고 짧다 | **모호함이 없어 기계 처리에 유리** |
| 이 실습 | 참고만 | ✅ **우리 데이터가 사용** |

<br>

> **⚠️ 모델은 데이터의 표기법을 배운다**
>
> 학습 데이터가 UCI이므로, **학습 후 모델도 UCI로 답하게 된다.**
> 만약 SAN으로 답하길 원한다면 **데이터를 SAN으로 바꿔야** 한다.
> 프롬프트로 "SAN으로 답해줘"라고 부탁하는 것보다 훨씬 확실하다.
>
> 💡 2-2에서 배운 원칙 그대로다 — **파인튜닝의 결과물은 데이터를 닮는다.**

#### ⑤ 기대치를 현실적으로 잡자

체스는 같은 국면에서 **좋은 수가 여러 개**인 게임이다.
그리고 데이터의 `expected_output`은 **"실제 대국에서 그 사람이 둔 수"** 일 뿐,
유일한 정답은 아니다.

```
   중반 국면의 평균 합법 수  :  약 30개
   무작위로 찍었을 때 정답률  :  약 3%
```

> 👉 따라서 **정답을 맞히지 못해도 실패가 아니다.**
> **진짜 관찰 포인트는 "지정한 JSON 형식으로 답하는가"** 이다.
>
> 장황한 설명이 **`{"next best move": "..."}`** 한 줄로 바뀌었다면 학습은 성공한 것이다.


In [ ]:
from datasets import load_dataset

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Fine-tuning "전" 베이스 모델의 답변을 기록 (Before)
#
# ★ 테스트 입력은 '학습 데이터와 똑같은 형식'이어야 한다.
#   임의로 만든 기보 문자열("1. e4 e5 ...")을 넣으면
#   모델이 학습 때 본 적 없는 형태가 되어 학습 효과를 확인할 수 없다.
#   → 실제 샘플을 그대로 가져와 쓴다.
#
# ★ 단, 학습에 쓴 앞 5,000건이 아니라 '그 뒤'에서 가져온다.
#   (1-1 챕터의 train/test 분리 원칙)
# ═══════════════════════════════════════════════════════════
NEXT_MOVE_TASK = 'write the best possible move'
TRAIN_SIZE = 5000

_all = load_dataset('Thytu/ChessInstruct', split='train')
_next = _all.filter(lambda r: NEXT_MOVE_TASK in r['task'])
test_sample = _next[TRAIN_SIZE]        # 학습 구간 밖의 첫 샘플

TASK_PROMPT     = test_sample['task']              # 학습 때 system에 들어간 문자열 그대로
test_input      = test_sample['input']             # JSON 형식 입력
expected_answer = test_sample['expected_output']   # 정답 (검증용)

# ★ 학습 데이터와 동일한 메시지 구조로 구성한다
#   학습: [system: task] [user: input] [assistant: 정답]
#   추론: [system: task] [user: input] → 모델이 assistant를 생성
#
#   ⚠️ 이 구조가 어긋나면 모델은 학습한 패턴을 발동시키지 못하고
#      베이스 모델의 성향(장황한 설명)으로 되돌아간다.
#      파인튜닝 실패 원인 1순위다.
messages = [
    {'role': 'system', 'content': TASK_PROMPT},
    {'role': 'user',   'content': test_input},
]

text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors='pt').to('cuda')
with torch.no_grad():
    # ⚠️ 실행 시 아래 경고가 뜨는데 정상이므로 무시해도 된다.
    #    "Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set."
    #
    #    [원인] 모델이 generation_config에 max_length=32768을 이미 갖고 있다.
    #           (32768 = Gemma-3의 컨텍스트 길이 32K)
    #           둘 다 설정되어 있으니 max_new_tokens를 쓰겠다는 안내일 뿐이다.
    #
    #    [두 값의 차이]
    #      max_length     : 입력 + 출력 '전체' 토큰 수의 상한
    #      max_new_tokens : '새로 생성할' 토큰 수      ← 우리가 원하는 것
    #
    #    [끄고 싶다면] 이 줄 위에 model.generation_config.max_length = None 을 추가
    #
    # 💡 50토큰이면 대략 200자다. 학습 전 모델은 이 안에 답을 못 끝내고 잘리는데,
    #    그 자체가 "짧게 답할 줄 모른다"는 증거가 된다.
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)

response_before = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
)

print('[과제]')
print(TASK_PROMPT)
print(f'\n[입력] {test_input[:160]} ...')
print(f'\n[정답] {expected_answer}')
print(f'\n[학습 전 응답] {response_before[:300]}')
print('\n→ 지정된 JSON 형식으로 답하지 못하는 것을 확인하라.')
print('  Fine-tuning 후 이 답변이 어떻게 바뀌는지 마지막 셀에서 비교한다.')

> **🔍 학습 전 응답, 이렇게 관찰하자**
>
> | 확인할 것 | 무엇을 의미하는가 |
> |---|---|
> | **JSON으로 답했는가?** | ⭐ 핵심. `{"next best move": "..."}` 형식이 나왔는가 |
> | **답변이 긴가?** | 지정된 형식으로 답하는 훈련을 받은 적이 없다는 뜻 |
> | **중간에 잘렸는가?** | 정상이다. `max_new_tokens=50` 제한 때문이며, 뒤집어 말하면 **50토큰으로 안 끝날 만큼 장황**하다는 증거다 |
> | **입력을 이해하기는 했는가?** | JSON 입력을 받고 무엇을 해야 하는지 파악했는가 |

<br>

> **⚠️ 실행 시 나타나는 경고문에 대하여**
>
> ```
> Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set.
> `max_new_tokens` will take precedence.
> ```
>
> **에러가 아니라 안내다.** 모델이 기본으로 갖고 있는 `max_length`(32768 = Gemma-3의 컨텍스트 32K)와
> 우리가 지정한 `max_new_tokens`(50)가 함께 있어서 뜨는 메시지이며,
> 경고문이 직접 말하듯 **우리가 지정한 50이 적용된다.**

---

> **⭐ 실제로 이런 답변이 나온다 — 형식을 아예 이해하지 못한다**
>
> ```
> Okay, let's analyze the chess position and determine the best possible move.
>
> **Current Position:**
>
> rnb8  r8  w8  w7  p8  p7  p6  p5  p
> ```
>
> 두 가지를 관찰할 수 있다.
>
> **① 우리가 요구한 형식을 무시했다**
>
> `{"next best move": "..."}` 로 답하라고 `system` 프롬프트에 명시했는데,
> 모델은 **"자, 분석해봅시다"** 로 시작하는 자유 서술을 내놓았다.
>
> **② 체스판을 그리려다 완전히 깨졌다**
>
> `rnb8 r8 w8 w7 p8 p7...` — 이건 **체스판도 아니고 아무 의미도 없는 문자열**이다.
> 모델은 "체스니까 판을 그려야겠다"고 판단했지만, **JSON으로 된 수순 배열을
> 판으로 변환하는 능력이 없어** 엉뚱한 출력을 만들어냈다.
>
> 👉 즉 지금 모델은 **입력을 이해하지도, 출력 형식을 지키지도 못하는** 상태다.

---

> **💡 이것이 4-1에서 배운 환각과 이어진다**
>
> 모델은 **"모르겠습니다"라고 말하지 않았다.**
> 대신 **그럴듯해 보이는 무언가를 자신 있게 생성**했다.
>
> | 환각의 성질 | 이번 출력에서 |
> |---|---|
> | **유창하다** | "Okay, let's analyze..." 자연스러운 도입 |
> | **형식만 그럴듯하다** | `**Current Position:**` 같은 마크다운 헤더까지 사용 |
> | **내용은 무의미하다** | 깨진 문자열을 체스판인 양 출력 |
>
> LLM은 *"내가 이걸 실제로 할 수 있는가"* 를 확인하지 않는다.
> **문장으로서 자연스러운 것**을 이어 붙일 뿐이다.
>
> 👉 이것이 **Fine-tuning이 필요한 직접적 증거**다.
> 지금 모델에게 부족한 것은 체스 지식이 아니라
> **"이 과제에서는 어떻게 답해야 하는가"** 이다.

---

> **📌 결과가 다르게 나와도 정상이다**
>
> 모델·라이브러리 버전에 따라 답변이 달라질 수 있다.
> 체스판을 그리려 하지 않고 그냥 설명만 할 수도, 엉뚱한 수를 제시할 수도 있다.
>
> **어느 쪽이든 판단 기준은 하나다.**
>
> ```
>    {"next best move": "..."} 형식으로 답했는가?
>
>    -> 아니라면, 학습이 필요하다는 증거다
> ```


---

## 2. Full Fine-tuning의 문제 — 메모리 폭발

### 2-1. Full Fine-tuning이란?

가장 직관적인 학습 방법: 모델의 **모든 파라미터를 업데이트**한다.

하지만 심각한 문제가 있다. 학습에는 **모델 가중치 + Gradient + Optimizer 상태**를 모두 GPU에 올려야 한다.

### 2-2. 학습 시 메모리 구성

1-2 챕터에서 배운 학습 4단계를 떠올려 보자.
**순전파 → 손실 → 역전파 → 업데이트.** 각 단계가 메모리를 요구한다.

```
[Full Fine-tuning 메모리 구성 — 1B 모델, FP32 기준]

모델 파라미터:     1B × 4바이트 =  4GB  ← 가중치 자체
Gradient:         1B × 4바이트 =  4GB  ← 역전파가 만드는 기울기
Optimizer(AdamW):  1B × 8바이트 =  8GB  ← momentum + variance (2개)
Activations:       배치 크기에 따라 가변    ← 순전파의 중간 계산 결과
─────────────────────────────────
합계:                           20GB 이상
```

| 항목 | 언제 생기나 | 1-2 챕터 대응 |
|---|---|---|
| **모델 파라미터** | 항상 | 가중치 w, 편향 b |
| **Activation** | 순전파 시 | 각 층의 중간 출력 (역전파에 필요해서 보관) |
| **Gradient** | 역전파 시 | `loss.backward()`가 만드는 기울기 |
| **Optimizer 상태** | 업데이트 시 | Adam의 관성·적응적 학습률 저장분 |

<br>

> **💡 Optimizer가 왜 8바이트나 쓰는가?**
>
> 1-2에서 배운 **Adam**을 떠올려 보자. Adam은 두 가지를 기억한다고 했다.
> **관성(momentum)** 과 **가중치별 적응적 학습률(variance)**.
>
> 이 두 값을 **파라미터마다 하나씩** FP32로 저장한다.
> 파라미터 1개당 4바이트 × 2 = **8바이트**.
> 즉 Optimizer 상태만으로 **모델 가중치의 2배** 메모리를 차지한다.
>
> 👉 "Adam이 SGD보다 메모리를 많이 쓴다"는 말이 여기서 나온다.

<br>

> **⚠️ Activation은 왜 '배치 크기에 따라 가변'인가?**
>
> 역전파를 하려면 순전파의 중간 결과를 **전부 기억하고 있어야** 한다.
> 그래서 배치가 크면 그만큼 저장할 것도 많아진다.
>
> ```
>    배치 2  →  Activation 적음
>    배치 32 →  Activation 16배
> ```
>
> 아래 코드는 이 값을 **"모델 크기의 2배"로 거칠게 추정**한다.
> 실제로는 배치 크기, 시퀀스 길이, 모델 구조에 따라 크게 달라지므로
> **정확한 값이 아니라 '자릿수 감각'을 잡는 용도**로 이해하자.

우리 GPU는 **16GB VRAM**이다. 1B 모델의 Full Fine-tuning에만 20GB 이상이 필요하니
**사실상 불가능**하다. 7B 모델이면? 100GB 이상이다.

> **📌 그럼 실무에서는 어떻게 Full FT를 하나?**
>
> | 기법 | 아이디어 |
> |---|---|
> | Mixed Precision | FP32 대신 FP16/BF16으로 계산 → 메모리 절반 |
> | Gradient Checkpointing | Activation을 버리고 필요할 때 **다시 계산** → 메모리↓ 시간↑ |
> | ZeRO / FSDP | 여러 GPU에 파라미터·Optimizer를 **나눠서** 저장 |
> | Gradient Accumulation | 작은 배치를 여러 번 누적 (아래 실습에서 사용) |
>
> 이걸 다 써도 결국 **GPU 여러 장**이 필요하다.
> 👉 **한 장으로 해결하는 방법**이 다음 챕터의 PEFT다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Full Fine-tuning에 필요한 메모리를 직접 계산
#
# [계산 공식]
# 모델:      파라미터 수 × 4바이트 (FP32)
# Gradient:  파라미터 수 × 4바이트
# Optimizer: 파라미터 수 × 8바이트 (AdamW = momentum 4B + variance 4B)
# Activation: 모델 크기의 ~2배 (거친 추정 — 배치/시퀀스에 따라 크게 다름)
#
# ⚠️ 이 계산은 '정확한 예측'이 아니라 '자릿수 감각'을 잡기 위한 것이다.
#    실제로는 Mixed Precision, Gradient Checkpointing 등으로 달라진다.
#    핵심 메시지는 "16GB로는 어림도 없다"는 것 하나다.
#
# [예상 결과]
# 1B 모델: ~20GB → 16GB GPU에서 불가능
# 7B 모델: ~100GB+ → A100 여러 장 필요
# ═══════════════════════════════════════════════════════════

# 💡 현재 model은 이미 4-bit로 로딩된 상태다.
#    numel()은 '파라미터 개수'를 세는 것이므로 양자화와 무관하게 동일하다.
#    (개수는 같고, 개당 저장 크기만 다르다)
total_params = sum(p.numel() for p in model.parameters())

def calc_full_ft_memory(params, label):
    """Full Fine-tuning에 필요한 총 메모리를 계산한다."""
    model_mem = params * 4 / 1024**3     # FP32 가중치 (4바이트)
    grad_mem = params * 4 / 1024**3      # Gradient (파라미터마다 1개씩)
    optim_mem = params * 8 / 1024**3     # AdamW (momentum + variance = 8바이트)
    act_mem = model_mem * 2              # Activation (추정: 모델의 ~2배)
    total = model_mem + grad_mem + optim_mem + act_mem
    print(f'\n[{label}] Full Fine-tuning 메모리 요구량:')
    print(f'  모델 파라미터:  {model_mem:6.1f} GB')
    print(f'  Gradient:     {grad_mem:6.1f} GB')
    print(f'  Optimizer:    {optim_mem:6.1f} GB (AdamW = momentum + variance)')
    print(f'  Activation:   {act_mem:6.1f} GB (추정)')
    print(f'  ──────────────────────')
    print(f'  합계:         {total:6.1f} GB')
    return total

print(f'현재 모델 파라미터: {total_params:,}개 ({total_params/1e9:.2f}B)')

mem_1b = calc_full_ft_memory(total_params, 'Gemma-3-1B')
mem_7b = calc_full_ft_memory(7_000_000_000, '7B 모델 (참고)')

print(f'\n─────────────────────────────')
print(f'우리 GPU VRAM: 16 GB')
print(f'Gemma-3-1B Full FT: {mem_1b:.1f} GB → {"❌ 불가능" if mem_1b > 16 else "✅ 가능"}')
print(f'7B 모델 Full FT: {mem_7b:.1f} GB → ❌ A100 2장 이상 필요')
print(f'\n→ Full Fine-tuning은 16GB GPU에서 사실상 불가능하다.')
print(f'  해결책: 전체가 아닌 "일부 파라미터만" 학습하는 PEFT → 다음 챕터')

---

## 3. LoRA의 원리 — 1%만 학습하는 방법

![image_A](https://i.ibb.co/kV5rS1KW/image-A.png)

### 3-1. PEFT란?

**PEFT (Parameter-Efficient Fine-Tuning)** = 전체 파라미터 중 **극소수만 학습**하는 기법

- 사전학습된 가중치는 **고정(freeze)** ❄️
- 적은 수의 **추가 파라미터(adapter)** 만 학습 🔥
- Full FT와 비슷한 성능을 달성하면서 메모리를 대폭 절감

> **💡 3-1 챕터에서 이미 해봤다**
>
> 전이학습의 **Linear Probing**을 떠올려 보자.
> ResNet의 특징 추출부를 `requires_grad = False`로 얼리고
> 마지막 분류층만 학습했다. **전체의 0.05%만 학습**했는데도 성능이 나왔다.
>
> | | 3-1 Linear Probing | 5-1 LoRA |
> |---|---|---|
> | 얼리는 것 | 특징 추출부 | **모델 전체** |
> | 학습하는 것 | 마지막 층 **교체** | **새 파라미터 추가** |
> | 공통점 | **대부분을 얼리고 일부만 학습** | |
>
> 👉 발상은 같고, **"어디에 무엇을 붙이느냐"** 가 다르다.

PEFT의 대표 기법이 바로 **LoRA (Low-Rank Adaptation)** 이다.

### 3-2. LoRA의 핵심 아이디어

LoRA의 출발점은 이 관찰이다.

> **"파인튜닝으로 생기는 가중치의 변화량(ΔW)은 생각보다 단순하다."**

수학 용어로는 **ΔW가 low-rank(저차원) 구조를 갖는다**고 표현한다.
쉽게 말하면, **큰 행렬처럼 보이지만 실제로는 적은 정보만 담고 있다**는 뜻이다.

그렇다면 **큰 행렬을 통째로 학습할 필요 없이, 작은 행렬 두 개로 표현하면** 된다.

> **💡 비유: 건물 리모델링**
>
> - **Full FT** = 건물 전체를 허물고 새로 짓기 (비용 막대)
> - **LoRA** = 건물은 그대로 두고, 필요한 부분만 덧붙이기 (비용 최소)

수식으로 표현하면:

```
기존:  h = W x                    (W: 사전학습 가중치, 전체 학습)
LoRA:  h = W x + (α/r) · B A x    (W: 고정 ❄️,  B·A만 학습 🔥)
        └──┬──┘   └─────┬─────┘
        원본 그대로    변화량(ΔW)을 두 행렬로 분해
```

### 3-3. B, A 행렬의 모양 — d와 r이 의미하는 것

```
┌─────────────────────────────────────────┐
│         W (사전학습 가중치)               │ ← 고정 ❄️
│              d × d                      │
└─────────────────────────────────────────┘
                    +
┌──────────┐   ┌──────────┐
│    B     │ × │    A     │  ← 학습 🔥
│  d × r   │   │  r × d   │
└──────────┘   └──────────┘
```

| 기호 | 의미 | 비고 |
|:---:|------|------|
| **d** | 원본 가중치 행렬의 **차원** | 모델마다 다르다 (수백~수천) |
| **r** | LoRA의 **rank** (분해할 차원) | 보통 8, 16, 32 |

**B × A**를 계산하면 `(d × r) × (r × d) = d × d` 로 **W와 같은 크기**가 된다.
즉 작은 행렬 두 개를 곱해서 **원본과 같은 크기의 "변화량"** 을 만들어내는 것이다.

하지만 실제로 저장·학습하는 파라미터는 `d × d`가 아니라 **`2 × d × r`** 뿐이다.

### 3-4. 숫자로 비교

예를 들어 **d = 4096, r = 8** 인 가중치 행렬 하나를 보자.

| 구분 | 계산 | 파라미터 수 | 비율 |
|------|------|:---:|:---:|
| **W (전체 학습)** | 4096 × 4096 | **16,777,216** | 100% |
| **B + A (LoRA)** | (4096 × 8) + (8 × 4096) | **65,536** | **0.39%** |

전체의 **0.4%만 학습**해도 충분한 성능을 달성할 수 있다.

> **📌 d 값은 모델마다 다르다**
>
> `d`는 모델의 hidden dimension이며 모델 크기에 따라 달라진다.
> 위 예시의 4096은 **7B급 모델에서 흔한 값**이고,
> 우리가 쓰는 1B급 모델은 이보다 작다.
>
> 💡 **정확한 값이 궁금하다면 직접 확인하면 된다.**
> ```python
> print(model.config)          # 모델 설정 전체
> print(model.config.hidden_size)   # d 값
> ```
> 아래 LoRA 적용 셀을 실행한 뒤 학습 파라미터 수와 대조해 보자.

### 3-5. ⭐ 왜 처음부터 망가지지 않는가 — 초기화의 비밀

여기서 자연스러운 의문이 하나 생긴다.

> **"W에 뭔가를 더한다면, 학습 시작하자마자 모델이 망가지는 것 아닌가?"**

LoRA는 이 문제를 **초기화**로 해결한다.

```
   A : 작은 랜덤 값으로 초기화 (가우시안)
   B : 전부 0으로 초기화        ← 핵심!

   따라서 학습 시작 시점에는
   B × A = 0  →  h = W x + 0 = W x

   즉 '원본 모델과 완전히 동일한 상태'에서 출발한다.
```

> **⭐ 이것이 LoRA 설계의 우아한 지점이다**
>
> - 학습 시작: 원본 모델 그대로 (성능 손실 0)
> - 학습 진행: B가 0에서 조금씩 벗어나며 변화량이 쌓인다
> - 학습 종료: 필요한 만큼만 원본을 수정한 상태
>
> 👉 **"기존 능력을 망가뜨리지 않으면서 새 능력을 더한다."**
> 3-1에서 배운 **파괴적 망각(Catastrophic Forgetting)** 을 구조적으로 완화하는 설계다.

### 3-6. LoRA 핵심 하이퍼파라미터

| 파라미터 | 의미 | 권장값 |
|---------|------|:---:|
| **r (rank)** | 분해 차원. 클수록 표현력↑, 메모리↑ | 8, 16, 32 |
| **lora_alpha** | adapter 출력의 **스케일링 계수** | r 또는 r × 2 |
| **target_modules** | LoRA를 적용할 레이어 | q_proj, v_proj 등 |
| **lora_dropout** | 과적합 방지 드롭아웃 | 0.0 ~ 0.1 |

<br>

> **💡 lora_alpha와 r의 관계 — 학습률이 아니라 '스케일링'이다**
>
> adapter 출력에 **α/r** 을 곱한다.
>
> ```
>    h = W x + (α/r) · B A x
>                └─┬─┘
>              스케일링 계수
> ```
>
> `r=8, alpha=16`이면 **16/8 = 2배**로 adapter의 영향력이 커진다.
>
> ⚠️ 이것은 **학습률(learning rate)과는 다른 개념**이다.
> 학습률은 "한 걸음의 크기"이고, α/r은 "adapter 출력의 세기"다.
>
> **왜 이렇게 나누는가?** r을 바꿔가며 실험할 때
> **α/r 비율을 유지하면 학습 강도를 비슷하게** 맞출 수 있기 때문이다.
> r을 8에서 16으로 올리면 alpha도 16에서 32로 올리는 관행이 여기서 나온다.

> **📌 r은 클수록 좋은가? — 아니다**
>
> | r | 표현력 | 메모리 | 과적합 위험 |
> |---|---|---|---|
> | 4~8 | 낮음 | 최소 | 낮음 |
> | 16~32 | 중간 | 보통 | 보통 |
> | 64+ | 높음 | 큼 | **높음** |
>
> LoRA 논문에서도 **r을 크게 키운다고 성능이 계속 오르지는 않는다**고 보고했다.
> **작은 r로 시작해서 부족하면 올리는 것**이 정석이다.
> (1-2의 "모델을 키우면 과적합 위험도 커진다"와 같은 맥락)

### 3-7. target_modules — 어디에 붙일 것인가

Transformer 블록은 크게 **Attention**과 **MLP**로 구성된다. (2-1 챕터 참고)

| 그룹 | 모듈 | 역할 |
|---|---|---|
| **Attention** | `q_proj`, `k_proj`, `v_proj`, `o_proj` | 토큰 간 정보 교환 (Q·K·V) |
| **MLP** | `gate_proj`, `up_proj`, `down_proj` | 토큰별 표현 강화 |

<br>

> **💡 원논문은 Attention에만 붙였다**
>
> LoRA 논문은 `q_proj`, `v_proj` 두 곳만 사용했다.
> 하지만 이후 연구에서 **MLP까지 포함하면 성능이 더 좋다**는 것이 확인되어,
> 현재는 **7개 모듈 전부에 적용하는 것이 사실상 표준**이 되었다.
>
> ⚠️ 물론 붙이는 곳이 많을수록 학습 파라미터도 늘어난다. **성능과 비용의 균형** 문제다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: LoRA를 모델에 적용하고 학습 파라미터 비율 확인
#
# [이 셀에서 하는 것]
# 1. LoRA 하이퍼파라미터(r, alpha, target_modules)를 설정
# 2. FastModel.get_peft_model()로 LoRA adapter를 모델에 삽입
# 3. 전체 파라미터 중 학습되는 비율을 확인
#
# [내부에서 일어나는 일]
# target_modules에 해당하는 모든 Linear 층 옆에
# B(d×r)와 A(r×d) 행렬 쌍이 새로 붙는다.
# 원본 가중치는 requires_grad=False로 얼려진다. (3-1의 Linear Probing과 동일)
#
# [확인 포인트]
# - 학습 비율이 1~2% 수준으로 나오면 정상이다.
# - 나머지 98% 이상의 사전학습 가중치는 고정(freeze)된 상태이다.
# - 이 비율이 곧 "메모리를 얼마나 아꼈는가"와 직결된다.
#
# ⚠️ 이 셀을 두 번 실행하면 LoRA 위에 LoRA를 또 붙이게 된다.
#    파라미터 비율이 이상하면 커널 재시작 후 처음부터 다시 실행하자.
# ═══════════════════════════════════════════════════════════

# ★ 핵심: LoRA 하이퍼파라미터 설정
r = 8             # rank: 분해 차원 (작을수록 경량, 8이 무난한 출발점)
lora_alpha = 16   # 스케일링: α/r = 16/8 = 2배로 adapter 영향력 증폭
lora_dropout = 0.0  # 드롭아웃 0 = 사용 안 함 (데이터가 충분하고 1에폭만 학습하므로)

# LoRA를 적용할 레이어들
# 💡 원논문은 q_proj, v_proj 두 개만 썼지만
#    현재는 Attention + MLP 전체에 붙이는 것이 표준이다.
target_modules = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',  # Attention 레이어 (Q, K, V, 출력)
    'gate_proj', 'up_proj', 'down_proj',       # MLP 레이어
]

# LoRA 적용 (Unsloth 최적화)
# 💡 표준 PEFT라면 LoraConfig 객체를 만들어 get_peft_model()에 넘겨야 하지만,
#    Unsloth는 인자를 직접 받아 처리한다.
model = FastModel.get_peft_model(
    model,
    r=r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
)

# ========== 학습 파라미터 비율 확인 ==========
# 💡 requires_grad=True인 파라미터 = 학습 대상 = LoRA의 B, A 행렬들
#    3-1 전이학습에서 얼린 파라미터를 셌던 것과 완전히 같은 코드다.
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
ratio = trainable / total * 100

print(f'총 파라미터:  {total:>15,}개')
print(f'학습 파라미터: {trainable:>15,}개')
print(f'학습 비율:    {ratio:.2f}%')
print(f'\n→ 전체의 약 {ratio:.1f}%만 학습한다. (나머지 {100-ratio:.1f}%는 고정)')
print(f'  챕터 2에서 계산한 Full FT 메모리 문제가 이것으로 해결된다.')

> **🔍 이 숫자가 의미하는 것**
>
> 챕터 2에서 계산한 Full FT 메모리와 비교해 보자.
>
> | | Full FT | LoRA |
> |---|---|---|
> | 학습 파라미터 | 전체 (100%) | **1~2%** |
> | Gradient 메모리 | 전체 크기만큼 | **1~2%만** |
> | **Optimizer 메모리** | 전체의 **2배** | **1~2%의 2배** |
>
> ⭐ **가장 큰 절감은 Optimizer에서 나온다.**
> Adam은 파라미터당 8바이트를 쓰는데, 학습 파라미터가 1%면 그 8바이트도 1%만 필요하다.
> 챕터 2에서 "Optimizer가 모델의 2배"라고 했던 그 부담이 거의 사라지는 것이다.

> **📌 LoRA adapter는 파일로 저장하면 얼마나 될까?**
>
> 학습이 끝나면 **B와 A 행렬만 저장**하면 된다. 원본 모델은 안 바뀌었으니까.
>
> ```
>    원본 모델      :  수 GB
>    LoRA adapter  :  수십 MB     ← 1/100 수준!
> ```
>
> 💡 **이것이 실무에서 LoRA가 사랑받는 또 다른 이유다.**
> 같은 베이스 모델 하나에 **고객사별 adapter를 여러 개** 붙였다 뗐다 할 수 있다.
> 모델을 통째로 복사할 필요가 없다.
>
> ```
>    베이스 모델 (공유)  +  adapter_고객A  →  고객A 전용 모델
>                       +  adapter_고객B  →  고객B 전용 모델
> ```

> **🧪 직접 해볼 실험 (수업 후)**
>
> `r` 값을 바꿔가며 학습 파라미터 비율이 어떻게 변하는지 확인해 보자.
> 커널을 재시작하고 `r = 16` 또는 `r = 32`로 바꿔 실행하면 된다.
>
> | r | 예상 학습 비율 | 확인할 것 |
> |---|---|---|
> | 4 | 약 절반 | 표현력이 부족한가 |
> | 8 | 기준 | — |
> | 32 | 약 4배 | 성능이 4배 좋아지는가? (그렇지 않다) |


---

## 4. QLoRA — 양자화 + LoRA의 결합

![image_B](https://i.ibb.co/Gf1yTfV3/image-B.png)

LoRA로 **학습할 파라미터**는 1%로 줄였다. 그런데 아직 문제가 남아 있다.

> **"학습은 1%만 한다지만, 원본 모델 100%는 여전히 GPU에 올려야 하지 않는가?"**

맞다. 얼려둔 가중치도 **순전파에는 사용**되므로 메모리에 있어야 한다.
7B 모델이면 FP16으로도 **14GB**다. 16GB GPU에서는 빠듯하다.

**이 마지막 문제를 푸는 것이 QLoRA다.**

### 4-1. 양자화(Quantization)란? — 지금 필요한 만큼만

> **➡️ 이 절은 '맛보기'다.** 양자화의 원리는 **다음 시간(5-2)** 에서 자세히 배운다.
> 여기서는 QLoRA를 이해하는 데 필요한 최소한만 다룬다.

**양자화** = 숫자를 표현하는 **비트 수를 줄여** 메모리를 아끼는 기법이다.

```
   원래 (FP16)  :  0.3721904...  →  16비트로 저장
   양자화 (INT4) :  0.375         →  4비트로 저장

   정밀도를 조금 잃는 대신, 용량이 1/4이 된다
```

| 정밀도 | 1개당 크기 | 1B 모델 | 7B 모델 |
|---|:---:|:---:|:---:|
| FP32 | 4바이트 | 4GB | 28GB |
| FP16 | 2바이트 | 2GB | 14GB |
| **INT4** | **0.5바이트** | **0.5GB** | **3.5GB** |

<br>

> **💡 비유: 사진 파일 압축**
>
> - **FP16** = 원본 RAW 파일. 화질은 최고지만 용량이 크다
> - **INT4** = 적당히 압축한 JPG. 자세히 보면 손실이 있지만 **알아보는 데는 문제없다**
>
> 모델도 마찬가지다. 가중치의 소수점 아래를 조금 뭉개도
> **전체적인 판단 능력은 대부분 유지**된다.

> **📌 5-2에서 배울 것 (지금은 이름만)**
>
> | 개념 | 한 줄 |
> |---|---|
> | **NF4** | 가중치가 정규분포를 따른다는 점을 이용한 4-bit 형식 |
> | **Double Quantization** | 양자화에 쓰인 상수까지 한 번 더 압축 |
> | **PTQ / QAT** | 학습 후 양자화 vs 학습 중 양자화 |
>
> 우리가 셀 3에서 쓴 `load_in_4bit=True`가 이 기술들을 자동으로 적용한 것이다.

### 4-2. QLoRA = 양자화된 모델 + LoRA Adapter

QLoRA는 두 가지를 **결합**한다.

```
┌─────────────────────────────────────────┐
│   Base Model (사전학습 가중치)            │ ← 4-bit로 압축 ❄️  (고정)
│   INT4 (~0.5GB per 1B params)           │
└─────────────────────────────────────────┘
                    +
┌──────────┐   ┌──────────┐
│    B     │ × │    A     │  ← 16-bit로 학습 🔥 (전체의 ~1%)
│  d × r   │   │  r × d   │
└──────────┘   └──────────┘
```

| 구분 | LoRA | QLoRA |
|------|:---:|:---:|
| Base Model | FP16 (~2GB/1B) | **INT4 (~0.5GB/1B)** |
| Adapter | FP16 | FP16 (동일) |
| 7B 모델 베이스 | ~14GB | **~3.5GB** |
| 16GB GPU에서 7B | ❌ 빠듯 | ✅ 여유 |

> **⭐ 핵심 질문: "4-bit로 압축했는데 학습이 되나?"**
>
> 가장 많이 묻는 지점이다. 답은 이렇다.
>
> ```
>    순전파 시 :  INT4 가중치를 잠깐 16-bit로 풀어서(dequantize) 계산
>    역전파 시 :  기울기는 16-bit로 계산되어 'Adapter에만' 흘러간다
>    업데이트   :  Adapter(B, A)만 갱신.  베이스 모델은 영원히 그대로 ❄️
> ```
>
> 즉 **INT4 가중치 자체는 학습되지 않는다.** 계산에만 참여할 뿐이다.
> 학습되는 것은 **16-bit로 유지되는 Adapter**뿐이므로 정밀도 문제가 없다.
>
> 👉 **"압축된 것은 얼려두고, 새로 붙인 것만 정밀하게 학습한다."**
> 이것이 QLoRA가 성립하는 이유다.

### 4-3. 세 가지 절감이 합쳐진다

챕터 2에서 계산한 메모리 항목을 다시 보자. QLoRA는 **세 곳을 동시에** 줄인다.

| 항목 | Full FT | QLoRA | 줄어든 이유 |
|---|:---:|:---:|---|
| 모델 가중치 | 100% | **~12%** | 4-bit 양자화 (FP32 대비) |
| Gradient | 100% | **~1%** | Adapter에만 발생 |
| Optimizer | 200% | **~2%** | Adapter 것만 저장 |

> ⭐ **혼자서는 부족하다. 둘이 합쳐져야 16GB에 들어간다.**
>
> - **양자화만** 쓰면 → 베이스는 작아지지만 Gradient·Optimizer가 여전히 폭발
> - **LoRA만** 쓰면 → 학습 부담은 줄지만 베이스 모델이 GPU를 차지
> - **QLoRA** → 양쪽을 동시에 해결

### 4-4. QLoRA의 한계도 알아두자

> **⚠️ 공짜는 없다**
>
> | 한계 | 설명 |
> |---|---|
> | **약간의 성능 손실** | 양자화로 정밀도가 떨어져 Full FT 대비 미세하게 낮을 수 있다 |
> | **추론 속도** | 순전파마다 dequantize가 필요해 FP16보다 **느릴 수** 있다 |
> | **하드웨어 의존** | bitsandbytes는 NVIDIA GPU 중심. 환경 제약이 있다 |
> | **지식 주입의 한계** | 형식·말투 학습에는 강하지만, **새로운 사실 주입은 RAG가 낫다** |
>
> 💡 마지막 항목이 중요하다. **1-2절의 판단 기준**을 다시 떠올리자.
> "알아야 하는가(RAG), 할 줄 알아야 하는가(Fine-tuning)."

### 4-5. Unsloth의 역할

| 구분 | 표준 QLoRA (HF + PEFT) | Unsloth QLoRA |
|------|:---:|:---:|
| 학습 속도 | 기준 | **약 2배 이상 빠름** |
| 메모리 | 기준 | **수십 % 절감** |
| 코드 복잡도 | `BitsAndBytesConfig` + `LoraConfig` + `PeftModel` | **`FastModel` 두 줄** |

Unsloth는 **알고리즘을 바꾼 것이 아니라 구현을 최적화**한 것이다.
QLoRA라는 기법 자체는 동일하며, GPU 커널을 직접 최적화해 속도를 끌어올렸다.


---

## 5. 실전: QLoRA 학습

![image_C](https://i.ibb.co/C33xtCxD/image-C.png)

이론은 끝났다. 이제 **실제로 모델을 학습**시킨다.

### 5-1. 학습 파이프라인 전체 흐름

```
① 데이터 로드 → ② Chat Template 변환 → ③ Trainer 설정 → ④ 학습 → ⑤ 저장/추론
```

> **💡 1-2 챕터의 학습 루프와 같은 구조다**
>
> | 1-2 (직접 구현) | 5-1 (Trainer가 대행) |
> |---|---|
> | `for epoch in range(...)` | `num_train_epochs=1` |
> | `optimizer.zero_grad()` | 자동 |
> | `loss.backward()` | 자동 |
> | `optimizer.step()` | 자동 |
>
> `SFTTrainer`는 우리가 손으로 짰던 학습 루프를 **한 번에 처리**해 준다.
> 내부에서 일어나는 일은 1-2에서 배운 그대로다.

### 5-2. 데이터셋: ChessInstruct

| 항목 | 내용 |
|------|------|
| 이름 | `Thytu/ChessInstruct` |
| 전체 규모 | **99,000건** |
| 구조 | `task`(지시), `input`(수순 JSON), `expected_output`(정답 JSON) |
| 사용량 | 5,000건 |

<br>

> **⚠️ 이 데이터셋은 여러 과제가 섞여 있다 — 반드시 필터링해야 한다**
>
> 99,000건 안에 최소 6종류의 과제가 들어 있다.
>
> | 과제 | 예시 |
> |---|---|
> | 최종 스코어 예측 | "이 대국의 최종 점수는?" |
> | 우세 판단 | "백과 흑 중 누가 유리한가?" |
> | **다음 최선 수 예측** ⭐ | **"다음에 둘 최선의 수는?"** ← 우리 목표 |
> | 빠진 수 채우기 | "누락된 수를 채워라" |
>
> **필터링하지 않으면 서로 다른 과제가 뒤섞여 학습된다.**
> 모델은 "무엇을 해야 하는지" 혼란스러워하고, 학습 효과가 크게 떨어진다.
>
> ```python
> NEXT_MOVE_TASK = 'write the best possible move'
> dataset = dataset.filter(lambda r: NEXT_MOVE_TASK in r['task'])
> # 99,000건 → 19,800건
> ```
>
> 👉 **데이터를 열어보지 않고 학습부터 돌리면 이런 문제를 놓친다.**
> 1-1 챕터의 EDA 정신이 파인튜닝에서도 그대로 유효하다.

> **📌 왜 5,000건만 쓰는가?**
>
> 수업 시간 안에 학습이 끝나야 하기 때문이다.
> 실무에서는 데이터가 많을수록 좋지만, **학습 시간도 비례해서 늘어난다.**
>
> ```
>    5,000건  →  약 15~20분
>    50,000건 →  약 3시간
> ```
>
> 💡 그리고 **데이터의 양보다 품질이 중요**하다.
> 2-2에서 배운 합성 데이터의 원칙 — **Garbage In, Garbage Out** — 이 여기서도 유효하다.

### 5-3. Chat Template이란?

LLM은 **모델마다 정해진 대화 형식**을 기대한다.
원본 데이터를 그 형식에 맞게 변환해야 학습이 제대로 된다.

```text
[원본 데이터]                                    [Chat Template 변환 후]
task: "...write the best possible move"    →    system: "...write the best possible move"
input: {"moves": ["e2e4", "e7e5", ...]}    →    user:   {"moves": ["e2e4", "e7e5", ...]}
expected_output: {"next best move":"d6d5"} →    assistant: {"next best move":"d6d5"}
```

⭐ **`assistant` 부분이 학습 대상**이다.
모델은 "이런 입력을 받으면 이런 JSON을 내놓는다"를 5,000번 반복 학습한다.

Gemma-3는 실제로 이런 형태의 토큰을 사용한다.

```bash
<start_of_turn>user
{"moves": ["e2e4", "e7e5", "g1f3", ...]}<end_of_turn>
<start_of_turn>model
{"next best move": "d6d5"}<end_of_turn>
```

> **⚠️ 형식이 틀리면 학습이 안 된다**
>
> 모델은 사전학습 때 **이 형식으로 대화를 배웠다.**
> 다른 형식으로 학습시키면 모델이 혼란스러워한다.
>
> 그래서 직접 문자열을 조립하지 않고 **`tokenizer.apply_chat_template()`** 에 맡긴다.
> 2-1에서 "토크나이저와 모델의 이름을 맞춰야 한다"고 했던 것과 같은 원리다.


In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 데이터 로드 + Chat Template 변환
#
# [이 셀에서 하는 것]
# 1. ChessInstruct 데이터셋에서 '다음 수 예측' 과제만 필터링 (99,000 -> 19,800건)
# 2. 그중 5,000건만 학습용으로 선택 (나머지는 평가용으로 남긴다)
# 3. 원본 3개 필드(task, input, expected_output)를
#    LLM이 이해하는 conversations 형식(system/user/assistant)으로 변환
# 4. Gemma-3의 chat template을 적용하여 실제 학습 텍스트로 변환
#
# [변환 흐름]
# 원본: {task: "...best possible move",
#        input: '{"moves": ["e2e4", "e7e5", ...]}',
#        expected_output: '{"next best move": "d6d5"}'}
#   ↓ convert_to_chatml()
# Chat: {conversations: [{role:system,...}, {role:user,...}, {role:assistant,...}]}
#   ↓ formatting_prompts_func()
# 텍스트: '<start_of_turn>user\n{"moves": [...]}<end_of_turn>'
#         '<start_of_turn>model\n{"next best move": "d6d5"}<end_of_turn>'
#
# ★ 입력·출력이 모두 JSON이라는 점에 주목.
#   모델은 "JSON을 받으면 JSON으로 답한다"는 패턴을 학습하게 된다.
#
# [add_generation_prompt=False인 이유]
# 학습 시에는 assistant 응답이 이미 포함되어 있으므로
# 모델이 새로 생성할 필요가 없다.
# 반대로 추론 시에는 True로 둬야 "이제 네가 답할 차례"라는 신호가 붙는다.
# (챕터 1의 ch1-before 셀에서 True를 쓴 이유)
#
# ⚠️ 이 셀은 학습 데이터를 만드는 단계다. 아직 학습은 시작되지 않는다.
#    출력된 샘플 텍스트를 꼭 눈으로 확인하자.
#    형식이 깨져 있으면 학습 자체가 무의미해진다.
# ═══════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════
# ⚠️ 이 데이터셋은 여러 과제가 섞여 있다 (99,000건 / 6종 이상)
#    - 최종 스코어 예측
#    - 우세 판단 (백/흑)
#    - ★ 다음 최선 수 예측        ← 우리 목표
#    - 빠진 수 채우기  등
#
#    필터링하지 않으면 서로 다른 과제가 뒤섞여 학습되어
#    모델이 "무엇을 해야 하는지" 혼란스러워한다.
# ═══════════════════════════════════════════════════════════
NEXT_MOVE_TASK = 'write the best possible move'

dataset = load_dataset('Thytu/ChessInstruct', split='train')
dataset = dataset.filter(lambda r: NEXT_MOVE_TASK in r['task'])
print(f'다음 수 예측 과제 필터링: {len(dataset):,}건')

# 앞 5,000건만 학습에 사용한다.
# ★ 뒤쪽은 평가용으로 남긴다 — 1-1에서 배운 train/test 분리 원칙.
#   학습에 쓴 데이터로 평가하면 "기출문제 외운 수험생에게 기출을 내는 것"과 같다.
TRAIN_SIZE = 5000
dataset = dataset.select(range(TRAIN_SIZE))
print(f'학습에 사용: {len(dataset):,}건 (나머지는 평가용)')
print(f'샘플: {dataset[0]}')

# ========== 1단계: Chat Template 변환 ==========
def convert_to_chatml(example):
    """원본 데이터를 conversations 형식(system/user/assistant)으로 변환한다.

    💡 세 역할의 의미
       system    : 모델에게 주는 역할·지시 (2-2의 Role Prompting)
       user      : 사용자 입력
       assistant : 모델이 내놓아야 할 정답  ← 이 부분을 학습한다
    """
    return {
        'conversations': [
            {'role': 'system', 'content': example['task']},      # 작업 지시
            {'role': 'user', 'content': example['input']},       # 체스 기보
            {'role': 'assistant', 'content': example['expected_output']},  # 정답 수
        ]
    }

# .map() : 데이터셋의 모든 행에 함수를 적용한다 (pandas의 apply와 비슷)
dataset = dataset.map(convert_to_chatml)

# ========== 2단계: 텍스트 포맷팅 ==========
def formatting_prompts_func(examples):
    """conversations를 Gemma-3 chat template 텍스트로 변환한다."""
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False,
            add_generation_prompt=False  # 학습 시: 응답이 이미 있으므로 False
        ).removeprefix('<bos>')  # 중복 BOS 토큰 제거
        # ⚠️ apply_chat_template이 문장 시작 토큰(<bos>)을 붙이는데,
        #    Trainer가 학습 시 또 붙이므로 여기서 미리 제거한다.
        #    (중복되면 모델이 혼란스러워한다)
        for convo in examples['conversations']
    ]
    return {'text': texts}

# batched=True : 여러 행을 한 번에 묶어서 처리 → 훨씬 빠르다
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'\n변환 완료. 샘플 텍스트:')
print(dataset[0]['text'][:500])


> **🔍 출력된 샘플 텍스트를 꼭 확인하자**
>
> `<start_of_turn>user` ... `<start_of_turn>model` 같은 **특수 토큰**이 보이는가?
> 이것이 Gemma-3가 기대하는 대화 형식이다.
>
> | 확인할 것 | 왜 |
> |---|---|
> | 필터링 건수가 **19,800건**인가 | 다른 과제가 섞이지 않았다는 증거 |
> | 특수 토큰이 제대로 붙었는가 | 형식이 깨지면 학습이 무의미해진다 |
> | `model` 뒤에 **`{"next best move": ...}`** 가 있는가 | ⭐ 이 부분이 학습 대상이다 |
> | `<bos>`가 중복되지 않았는가 | 다음 셀에서 Trainer가 또 붙인다 |
>
> 💡 **학습을 시작하기 전에 데이터를 눈으로 보는 습관**을 들이자.
> 잘못된 데이터로 20분을 학습시키고 나서 발견하면 그 시간은 그냥 버려진다.
> 1-1 챕터의 EDA 정신이 여기서도 유효하다.

> **⭐ 이 실습에서 실제로 겪은 일**
>
> 처음 이 노트북을 만들 때는 데이터 형식을 확인하지 않고
> **"1. e4 e5 2. Nf3" 같은 SAN 문자열**로 테스트했다. 결과는 실패였다.
>
> ```
>    학습 데이터  :  {"moves": ["e2e4", ...]}  (JSON + UCI)
>    테스트 입력  :  "1. e4 e5 2. Nf3"          (문자열 + SAN)
>                        ↑ 모델이 처음 보는 형태
> ```
>
> 모델은 학습한 패턴을 발동시키지 못하고 **베이스 모델의 장황한 설명으로 되돌아갔다.**
>
> 👉 **파인튜닝 실패 원인 1순위가 이것이다.**
> **학습 때 쓴 형식과 추론 때 쓰는 형식은 반드시 같아야 한다.**


In [ ]:
# ⏱️ [학습 시간 안내 — 이 셀은 오래 걸린다!]
# - 강의장 환경(16GB VRAM): 약 20~30분 소요 예상
# → 학습이 진행되는 동안 앞의 이론(챕터 3~4)을 다시 읽으며 기다리면 된다.
#
# ⚠️ 실행 중에는 다른 셀을 돌리지 말 것. GPU 메모리가 부족해질 수 있다.

from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: SFTTrainer 설정 + 학습 실행
#
# [이 셀에서 하는 것]
# 1. SFTTrainer(지도 학습 미세조정 트레이너)를 설정한다.
# 2. train_on_responses_only()로 "답변 부분만" 학습하도록 설정한다.
# 3. trainer.train()으로 실제 학습을 실행한다.
#
# [SFTTrainer란?]
# Supervised Fine-Tuning(지도 학습 미세조정) 전용 트레이너이다.
# trl(Transformer Reinforcement Learning) 라이브러리에서 제공하며,
# LoRA/QLoRA와 자연스럽게 연동된다.
#
# [train_on_responses_only — 왜 필요한가?]
# 데이터에는 "질문(instruction)"과 "답변(response)"이 모두 포함되어 있다.
# 우리가 학습시키고 싶은 것은 "답변을 잘 하는 법"이지,
# "질문을 잘 읽는 법"이 아니다.
# train_on_responses_only()는 답변 부분의 Loss만 계산하여 학습 효율을 높인다.
#
# [주요 학습 설정 해설]
# - per_device_train_batch_size=2: GPU당 한 번에 처리하는 데이터 수
#     1-2에서 배운 '미니배치'다. 크면 안정적이지만 메모리를 많이 쓴다.
# - gradient_accumulation_steps=4: 4번 누적 후 업데이트 → 실효 배치 크기 = 2×4 = 8
#     ★ 메모리가 부족할 때 배치를 키우는 트릭.
#       기울기를 4번 모았다가 한 번에 업데이트하므로,
#       메모리는 배치 2만큼만 쓰면서 효과는 배치 8과 비슷해진다.
# - learning_rate=5e-5: 1-2에서 배운 '보폭'.
#     ⚠️ 파인튜닝은 이미 좋은 상태에서 출발하므로 학습률을 작게 잡는다.
#       3-1의 '파괴적 망각'과 같은 이유다.
# - num_train_epochs=1: 전체 데이터를 1번만 학습 (수업 시간 고려)
# - fp16/bf16: 학습 중 16비트 정밀도 사용 (메모리 절감 + 속도 향상)
#     bf16은 최신 GPU(Ampere 이상)에서 지원하며 fp16보다 수치적으로 안정적이다.
# - logging_steps=10: 10 step마다 Loss를 출력 (학습 진행 확인)
# - seed=42: 재현성 확보. 같은 시드면 같은 결과가 나온다.
#
# [학습 결과 해석 가이드]
# 학습이 완료되면 다음 값이 출력된다:
# - 학습 후 메모리: 16GB VRAM의 상당 부분 사용 (정상)
# - Loss: 값이 낮을수록 정답에 가깝다는 뜻
#
# ⚠️ Loss의 '절대값'보다 '추세'가 중요하다.
#    logging_steps=10마다 찍히는 Loss가 '점차 줄어들고 있는가'를 보자.
#    체스는 같은 상황에서 좋은 수가 여러 개이므로 Loss가 0에 수렴하지 않는다.
#
# 💡 Loss가 전혀 안 줄어든다면?
#    → 학습률이 너무 작거나, 데이터 형식이 잘못되었을 가능성이 높다.
#      (1-2에서 배운 학습률 디버깅 감각을 떠올리자)
# ═══════════════════════════════════════════════════════════

# Trainer 설정
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='text',
        output_dir='outputs',
        per_device_train_batch_size=2,      # GPU당 배치 크기
        gradient_accumulation_steps=4,       # 4번 누적 → 실효 배치 8
        learning_rate=5e-5,                  # 학습률
        num_train_epochs=1,                  # 1에폭 (이론 실습용)
        max_seq_length=max_seq_length,       # 시퀀스 길이 제한
        fp16=not torch.cuda.is_bf16_supported(),  # GPU가 bf16 미지원이면 fp16
        bf16=torch.cuda.is_bf16_supported(),      # GPU가 bf16 지원하면 bf16
        logging_steps=10,                    # 10 step마다 Loss 출력
        seed=42,
    ),
)

# ★ 핵심: Response(답변) 부분만 학습하도록 설정
# instruction_part: 질문이 시작되는 토큰 (학습 제외 → Loss 계산 안 함)
# response_part: 답변이 시작되는 토큰 (학습 대상 → Loss 계산)
#
# 💡 왜 이렇게 하는가?
#    질문 부분까지 학습하면 모델이 '질문을 생성하는 법'까지 배운다.
#    우리가 원하는 것은 '주어진 질문에 답하는 법'이므로,
#    답변 부분에만 Loss를 걸어야 학습이 효율적이다.
#    ⚠️ 토큰 문자열이 모델의 chat template과 정확히 일치해야 동작한다.
trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)

# ========== 학습 실행 ==========
print('⏱️ 학습 시작! (RTX 5060 Ti 기준 약 20~30분 소요)')
print('   학습이 진행되는 동안 logging_steps=10마다 Loss가 출력된다.')
print('   Loss가 점차 감소하면 학습이 잘 되고 있는 것이다.\n')

if torch.cuda.is_available():
    start_mem = torch.cuda.max_memory_reserved() / 1024**3
    print(f'학습 전 메모리: {start_mem:.2f} GB')

trainer_stats = trainer.train()

if torch.cuda.is_available():
    end_mem = torch.cuda.max_memory_reserved() / 1024**3
    print(f'\n학습 후 메모리: {end_mem:.2f} GB')
    print(f'학습에 사용된 메모리: {end_mem - start_mem:.2f} GB')

# [결과 해석]
# Loss: 값이 낮을수록 모델이 정답에 가깝게 학습된 것이다.
#   체스는 같은 상황에서 여러 좋은 수가 있으므로 0.5~0.8이면 충분하다.
# 메모리: 16GB 중 ~12GB 사용은 정상 범위이다.
#   QLoRA 덕분에 16GB GPU에서 학습이 가능했다는 것이 핵심이다.
print(f'\n학습 완료! Loss: {trainer_stats.training_loss:.4f}')
print(f'→ Loss가 0.5~0.8 범위이면 정상적으로 학습된 것이다.')
print(f'  (체스는 최선의 수가 여러 개이므로 Loss가 0에 수렴하지 않는다)')


In [ ]:
import json as _json

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Fine-tuning 전후 비교 — Before vs After
#
# [이 셀에서 하는 것]
# 챕터 1에서 기록한 학습 전 응답(response_before)과
# 학습 후 응답(response_after)을 나란히 비교한다.
#
# [확인 포인트]
# - 학습 전: 긴 설명문. 형식을 지키지 못하고 체스판을 그리려다 깨진 문자열 출력
# - 학습 후: {"next best move": "xxxx"} 한 줄
#
# ★ 판정은 세 층으로 나눈다 (아래 출력 참고)
#     ① 형식(JSON)  ← 우리가 실제로 학습시킨 것.  이것만 되면 성공
#     ② 간결성       ← 장황한 설명이 사라졌는가
#     ③ 정답 일치    ← 학습시킨 적 없는 것.  보너스
#
# ⚠️ ③이 틀려도 실패가 아니다.
#    체스는 한 국면에 합법 수가 평균 30개 내외이고,
#    expected_output은 "실제 대국에서 그 사람이 둔 수"일 뿐 유일한 정답이 아니다.
#    무작위로 찍으면 약 3%가 맞는다.
#
# [이 결과가 의미하는 것]
# 전체 파라미터의 ~1%만 학습했는데도 모델의 '출력 행동'이 완전히 바뀌었다.
# 16GB GPU에서 수십 분 만에 도메인 특화 모델을 만든 것이며,
# Full Fine-tuning(20GB+ 필요)으로는 아예 불가능했을 작업이다.
#
# ⚠️ 단, 1건만 보고 판단하면 안 된다.
#    반드시 다음 셀의 '다건 평가'까지 실행해서 확인할 것.
# ═══════════════════════════════════════════════════════════

# 학습 후 추론 (같은 입력으로)
# 💡 text는 챕터 1(ch1-before)에서 만든 그 변수다.
#    같은 입력을 써야 학습 전후를 공정하게 비교할 수 있다.
# ⚠️ 챕터 1 셀을 실행하지 않았다면 여기서 NameError가 난다.
inputs = tokenizer(text, return_tensors='pt').to('cuda')
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)

response_after = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
)


print('Fine-tuning 전후 비교')
print('=' * 70)
print(f'[입력] {test_input[:160]} ...')
print(f'\n[정답] {expected_answer}')
print('=' * 70)
print(f'\n[학습 전] {response_before}')
print('-' * 70)
print(f'\n[학습 후] {response_after[:600]}')
print('=' * 70)

# ========== 정량 검증 ==========
# ★ 이전에는 "a6가 나왔나?"를 눈대중으로 봤지만,
#   이제 정답 데이터가 있으므로 문자열 비교로 정확히 판정할 수 있다.
try:
    gold = list(_json.loads(expected_answer).values())[0]   # 키 이름과 무관하게 값만 추출
except Exception:
    gold = expected_answer.strip()

is_json   = 'next best move' in response_after
is_hit    = gold in response_after
is_short  = len(response_after.strip()) < 80

print(f'\n── 결과 판정 ──')
print(f'  ① 형식(JSON)  : {"✅" if is_json  else "❌"}   ← 우리가 학습시킨 것')
print(f'  ② 간결성       : {"✅" if is_short else "❌"}   (80자 미만)')
print(f'  ③ 정답 일치    : {"✅" if is_hit   else "❌"}   (정답: {gold})  ← 보너스')

# 학습 효율 요약
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'\n── QLoRA 학습 효율 요약 ──')
print(f'  학습 파라미터: {trainable:,}개 (전체의 {trainable/total*100:.2f}%)')
print(f'  사용 GPU: 16GB VRAM (Full FT라면 20GB+ 필요)')
print(f'  핵심: 1%의 파라미터만 학습해도 모델의 출력 형식이 바뀐다')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ⭐ 다건 평가 — 반드시 실행할 것 (1건만 보고 판단하면 안 된다)
#
# [왜 필요한가]
# 앞 셀은 '단 1건'의 결과다. 운 좋게 맞았을 수도, 운 나쁘게 틀렸을 수도 있다.
# 잘 나온 사례 하나만 골라 보여주는 것을 '체리피킹(cherry-picking)'이라 한다.
# 실제로 앞 셀의 데모 샘플(_next[5000])은 아래 평가 구간에 포함되어 있으므로,
# 두 결과를 비교하면 그 1건이 대표성이 있는지 바로 알 수 있다.
#
# [두 지표를 분리해서 측정한다]
#   형식 준수율  : JSON으로 답했는가        <- 우리가 학습시킨 것
#   정답 일치율  : 그 수가 정답과 같은가     <- 학습시킨 적 없는 것
#
# [해석 기준]
#   형식 준수율  : 100%에 가까워야 학습 성공
#   정답 일치율  : 무작위(약 3%)와 비교할 것.
#                 ⚠️ 20건은 통계적으로 매우 적다.
#                    1/20의 95% 신뢰구간은 0.9% ~ 23.6% 로,
#                    무작위와 구분되지 않는다.
#                    제대로 측정하려면 200건 이상이 필요하다.
#
# ⏱️ 20건에 약 1~2분 소요. N_EVAL을 늘리면 그만큼 오래 걸린다.
# ═══════════════════════════════════════════════════════════
N_EVAL = 20
hits = fmt_ok = 0

for i in range(TRAIN_SIZE, TRAIN_SIZE + N_EVAL):
    s = _next[i]      # 학습에 쓰지 않은 구간에서 가져온다 (train/test 분리)

    # ★ 학습 데이터와 동일한 메시지 구조로 구성 (형식 정렬)
    msgs = [{'role': 'system', 'content': s['task']},
            {'role': 'user',   'content': s['input']}]
    t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(t, return_tensors='pt').to('cuda')
    with torch.no_grad():
        # 30토큰이면 JSON 한 줄에 충분하다 (평가 속도를 위해 짧게)
        out = model.generate(**inp, max_new_tokens=30, do_sample=False)
    resp = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

    g = list(_json.loads(s['expected_output']).values())[0]   # 키 이름과 무관하게 값만 추출
    if 'next best move' in resp: fmt_ok += 1    # 형식 준수 여부
    if g in resp:                hits    += 1    # 정답 일치 여부

print(f'평가 {N_EVAL}건 (학습에 쓰지 않은 데이터)')
print(f'  형식 준수율 : {fmt_ok}/{N_EVAL}  ({fmt_ok/N_EVAL*100:.0f}%)   ← 학습 목표')
print(f'  정답 일치율 : {hits}/{N_EVAL}  ({hits/N_EVAL*100:.0f}%)   ← 보너스 (무작위 ≈ 3%)')

> **🎯 결과 판정 — 세 층으로 나눠 읽는다**
>
> | # | 판정 항목 | 이번 결과 | 의미 |
> |:--:|---|:---:|---|
> | ① | **형식 (JSON)** | **20/20 · 100%** | ⭐ 우리가 학습시킨 것 |
> | ② | **간결성** | ✅ | 장황한 설명이 사라짐 |
> | ③ | **정답 일치** | 1/20 · 5% | 학습시킨 적 없는 것 |
>
> ⭐ **①이 100%라는 것 — 이것이 오늘 실습의 성공 지표다.**
> 우리는 5,000건으로 **"이 과제에는 이런 형식으로 답한다"** 를 가르쳤고, 완벽하게 학습됐다.

---

> **✅ 무엇이 바뀌었나 — Before/After 직접 대조**
>
> **학습 전**
> ```
> Okay, let's analyze the chess position and determine the best possible move.
>
> **Current Position:**
> ```
> rnb8  r8  w8  w7  p8  p7  p6  p5  p
> ```
>
> 체스판을 ASCII로 그리려다 **완전히 깨진 문자열**을 뱉었다.
> **무엇을 요구받았는지조차 파악하지 못한** 상태다.
>
> **학습 후**
> ```
> {"next best move": "d6d5"}
> ```
>
> 한 줄. 정확한 JSON. 좌표식(UCI) 표기.
>
| 관점 | 학습 전 | 학습 후 |
|---|---|---|
| 출력 형식 | 자유 서술 + 깨진 ASCII | **`{"next best move": "..."}"`** |
| 길이 | 128토큰을 다 써도 안 끝남 | **25자** |
| 과제 이해 | 못 함 | **정확히 이해** |
>
> ⚠️ 바뀐 것은 **모델 코드가 아니다.** 전체 파라미터의 **0.97%**, 6,522,880개만 학습했을 뿐이다.

---

> **⭐ 가장 중요한 관찰 — 데모 1건에 속지 마라**
>
> 위 셀에서 **정답까지 맞혔다**(`d6d5`). 그런데 20건으로 넓혀보니 **1/20**이었다.
>
> ```
>    데모 샘플     : _next[5000]           -> 정답 ✅
>    평가 구간     : _next[5000 ... 5019]  -> 1/20 정답
>                        ↑
>    ⭐ 20건 중 유일하게 맞힌 그 1건이, 바로 우리가 데모로 본 그 샘플이다
> ```
>
> **1건만 보고 "정답까지 맞히는 모델을 만들었다"고 했다면 완전히 틀린 결론**이었다.
>
> 👉 **이것이 체리피킹(cherry-picking)이다.**
> 잘 나온 사례 하나를 골라 보여주면 누구나 대단해 보인다.
> AI 제품 데모나 논문 결과를 볼 때 항상 물어야 할 질문이 있다.
>
> > **"몇 건으로 측정했는가? 실패 사례는 어디 있는가?"**
>
> 💡 1-1 챕터에서 **"정확도 하나만 보지 말고 클래스별로 쪼개보라"** 고 했던 것과 같은 태도다.

---

> **📊 정답률 5%를 정직하게 해석하기**
>
> 낮아 보이는가? **기준선과 비교해야 판단할 수 있다.**
>
> | 기준 | 정답률 |
> |---|:---:|
> | 무작위로 찍기 (합법 수 평균 약 32개) | **약 3%** |
> | **우리 모델 (20건 평가)** | **5%** |
> | 체스 감각을 학습한 모델 | 20% 이상 |
>
> ⚠️ **그런데 20건은 판단하기에 너무 적다.**
> 1/20의 95% 신뢰구간은 **0.9% ~ 23.6%** 다. 무작위(3%)와 **통계적으로 구분되지 않는다.**
>
> ```
>    n=20  ->  신뢰구간이 너무 넓어 "찍기보다 나은지" 말할 수 없다
>    n=200 이상이어야 의미 있는 비교가 가능하다
> ```
>
> 👉 **정직한 결론: "정답률은 측정하기에 표본이 부족하며, 현재로선 찍기 수준과 구분되지 않는다."**
>
> 💡 실무에서 **"정확도 5% 향상"** 같은 주장을 볼 때도 똑같이 물어야 한다.
> **표본이 몇 개였는가?** 1-1에서 배운 **교차검증의 표준편차**를 함께 보라던 이유가 이것이다.

---

> **🤔 왜 정답률은 낮은가 — 당연한 결과다**
>
> **우리는 체스를 가르친 적이 없다.**
>
> | | 무엇을 가르쳤나 | 결과 |
> |---|---|:---:|
> | **형식** | "JSON 한 줄로 답해라" | **100%** ✅ |
> | **체스 실력** | 가르친 적 없음 | 5% |
>
> 다음 수 예측은 **약 32개의 합법 수 중 하나를 고르는 문제**다.
> 이걸 잘하려면 국면 평가, 전술 계산, 수읽기가 필요한데,
> **5,000건 · 1에폭 · 파라미터 0.97%** 로는 불가능하다.
>
> 참고로 실제 체스 엔진들은 **수천만 판**을 학습하거나 **탐색 알고리즘**을 함께 쓴다.
>
> ⭐ **이것이 1장에서 배운 원칙을 실증한다.**
>
> ```
>    Fine-tuning  ->  "할 줄 알게" 만든다  (형식·말투·행동 양식)   ✅ 성공
>    RAG          ->  "알게" 만든다        (사실 정보)
>
>    ⚠️ Fine-tuning으로 '지식'을 주입하려는 시도는 대체로 비효율적이다
> ```
>
> 👉 만약 진짜 강한 체스 AI를 만들어야 한다면? **Fine-tuning이 아니라 다른 접근**이 필요하다.
> 4-2에서 배운 **Tool-use**로 체스 엔진을 도구로 붙이는 편이 훨씬 낫다.

---

> **✅ 그래서 오늘 실습은 성공인가? — 그렇다**
>
> | 확인한 것 | 근거 |
> |---|---|
> | 16GB GPU 한 장으로 LLM 학습이 가능하다 |  Out Of Memory 없이 완주 |
> | **1%만 학습해도 모델의 행동이 바뀐다** | 형식 준수율 0% → **100%** |
> | 파인튜닝 결과는 **데이터를 닮는다** | JSON·UCI 표기를 그대로 습득 |
> | **프롬프트 정렬이 결정적이다** | 형식을 맞추기 전에는 학습 효과가 거의 없었다 |
> | Fine-tuning은 **형식을 가르치지 지식을 주지 않는다** | 형식 100% vs 정답 5% |
>
> ⭐ 마지막 두 항목은 **교과서에 잘 안 나오지만 실무에서 가장 자주 겪는 문제**다.

---

> **🔬 더 개선하려면 — 무엇을 바꿀 것인가**
>
> **목표에 따라 조치가 완전히 다르다.**
>
> **(A) 형식 안정성을 더 높이고 싶다면** — 이미 100%라 개선 여지가 적다
>
> | 조치 | 기대 |
> |---|---|
> | 더 어려운 입력으로 평가 | 100%가 유지되는지 확인 |
> | `max_new_tokens` 축소 | 불필요한 생성 차단 |
>
> **(B) 정답률을 올리고 싶다면** — 여기가 진짜 과제다
>
> | # | 조치 | 비용 | 기대 |
> |:--:|---|:---:|---|
> | 1 | **평가 건수를 200건으로** | 10분 | ⭐ 먼저 **제대로 측정**해야 한다 |
> | 2 | 데이터를 19,800건 전부 사용 | 학습 ×4 | 패턴 학습 강화 |
> | 3 | 에폭을 2~3으로 | 시간 ×2~3 | — |
> | 4 | `r`을 16~32로 | 메모리 ↑ | 표현력 확보 |
> | 5 | 더 큰 모델 (4B 이상) | VRAM ↑ | 추론 능력 자체가 다름 |
> | 6 | **체스 엔진을 Tool로 연결** | — | ⭐ 근본적으로 다른 접근 (4-2) |
>
> ⭐ **1번을 가장 먼저 하라.** 측정이 부정확한 상태에서 개선을 시도하면
> 좋아졌는지 나빠졌는지 알 수 없다. **측정 없이는 개선도 없다.**

---

> **📌 학습한 모델은 어떻게 저장하나?**
>
> ```python
> # ① Adapter만 저장 (수십 MB) — 가장 일반적
> model.save_pretrained('chess_lora_adapter')
> tokenizer.save_pretrained('chess_lora_adapter')
>
> # ② 베이스 모델과 병합해서 저장 (수 GB) — 배포용
> # model.save_pretrained_merged('chess_model', tokenizer)
> ```
>
> | | Adapter만 | 병합 저장 |
> |---|---|---|
> | 크기 | **수십 MB** | 수 GB |
> | 사용 조건 | 같은 베이스 모델 필요 | **단독으로 완결** |
> | 적합 | 여러 어댑터를 갈아 끼울 때 | 배포·서빙 |
>
> ⚠️ ①로 저장했다면 불러올 때도 **반드시 같은 베이스 모델**을 써야 한다.
> Adapter는 특정 모델의 특정 층에 붙는 것이라, 다른 모델에는 붙지 않는다.
>
> 👉 자기주도 실습에서 직접 다뤄본다.

---

## 6. 정리

### 오늘 배운 전체 흐름

```
① Fine-tuning이 필요한 이유 — 사전학습 모델은 특정 형식·도메인에 약하다 (챕터 1)
② Full FT의 문제 — 모든 파라미터 학습 → 메모리 폭발 (챕터 2)
③ LoRA — 변화량(ΔW)을 작은 행렬 두 개로 분해, 1%만 학습 (챕터 3)
④ QLoRA — 베이스 모델을 4-bit로 압축 + LoRA → 16GB GPU에서도 학습 (챕터 4)
⑤ 실전 학습 — 데이터 변환 → Trainer → 학습 → 전후 비교 (챕터 5)
```

### 핵심 개념 요약

| 개념 | 한 줄 정리 |
|------|----------|
| **Fine-tuning** | 사전학습 모델을 특정 작업·형식에 맞게 추가 학습 |
| **Full FT** | 모든 파라미터 학습. 성능은 좋지만 메모리 폭발 |
| **PEFT** | 극소수 파라미터만 학습하는 기법의 총칭 |
| **LoRA** | ΔW를 저차원 행렬 B×A로 분해. **B는 0으로 초기화**해 원본을 보존 |
| **QLoRA** | 4-bit 양자화 + LoRA. 베이스는 얼리고 Adapter만 16-bit로 학습 |
| **Unsloth** | 최적화된 QLoRA 구현체. 알고리즘이 아니라 **속도**를 개선 |

### ⭐ RAG vs Fine-tuning — 언제 무엇을 쓰는가

이번 챕터에서 가장 중요한 판단 기준이다.

| | RAG (4-1) | Fine-tuning (5-1) |
|---|:---:|:---:|
| 비유 | 자료를 **쥐여준다** | 사람을 **교육시킨다** |
| 잘하는 것 | **사실 정보** | **형식·말투·행동 양식** |
| 정보가 바뀌면 | 문서만 교체 | 재학습 |
| 출처 추적 | ✅ | ❌ |

```
   "우리 회사 환불 규정이 뭐야?"      →  알아야 한다     →  RAG
   "우리 회사 말투로 답변해줘"        →  할 줄 알아야    →  Fine-tuning
```

> 💡 그리고 순서가 있다: **프롬프팅 → RAG → Fine-tuning.**
> 앞의 것으로 해결되면 뒤로 갈 이유가 없다.

### 🐛 자주 만나는 문제

| 증상 | 원인 | 해결 |
|---|---|---|
| `CUDA out of memory` | 배치·시퀀스 길이가 큼 | `batch_size` 또는 `max_seq_length` 축소 |
| `NameError: text / response_before` | 챕터 1 셀 미실행 | 위에서부터 순서대로 실행 |
| 학습 비율이 이상함 | LoRA 셀을 두 번 실행 | 커널 재시작 후 처음부터 |
| Loss가 줄지 않음 | 학습률이 너무 작거나 데이터 형식 오류 | 샘플 텍스트 확인 → `learning_rate` 조정 |
| 학습 후에도 설명문만 나옴 | **추론 프롬프트가 학습 형식과 다름** | ⭐ `system`+`user` 구조를 학습 데이터와 일치시킬 것 |
| 여러 과제가 섞여 학습됨 | `task` 필터링 누락 | `filter()`로 목표 과제만 선별 |
| 정답률이 낮음 | **학습시킨 적 없는 능력** | 정상. 형식 준수율을 볼 것 |
| `ModuleNotFoundError` | 설치 후 커널 미재시작 | 런타임 재시작 |

### 📊 이번 실습의 실측 결과

| 지표 | 결과 | 해석 |
|---|:---:|---|
| **형식 준수율** | **20/20 · 100%** | ⭐ 학습 목표 달성 |
| 정답 일치율 | 1/20 · 5% | 무작위(≈3%)와 통계적으로 구분 안 됨 |
| 학습 파라미터 | 6,522,880개 · **0.97%** | 전체의 1% 미만 |
| 사용 VRAM | 16GB | Full FT는 20GB+ 필요 |

<br>

> **⭐ 이 두 숫자가 오늘의 결론을 증명한다**
>
> ```
>    형식 100%  +  정답 5%
>
>    ->  Fine-tuning은 '형식(행동 양식)'을 가르친다
>        Fine-tuning은 '지식'을 주지 않는다
> ```
>
> 1-2절에서 배운 판단 기준이 그대로 실증되었다.
> **"알아야 하는가(RAG) / 할 줄 알아야 하는가(Fine-tuning)."**
>
> 💡 만약 진짜 강한 체스 AI가 필요하다면?
> 파인튜닝이 아니라 **4-2에서 배운 Tool-use**로 체스 엔진을 붙이는 편이 훨씬 낫다.
> **기법을 목적에 맞게 고르는 것**이 실력이다.

### 🔬 직접 해볼 실험

| # | 실험 | 바꿀 것 | 관찰할 것 |
|:--:|---|---|---|
| 0 | **평가 건수 확대** | `N_EVAL = 200` | ⭐ **먼저 제대로 측정하라.** 측정 없이는 개선도 없다 |
| 1 | rank 조절 | `r = 8` → `16`, `32` | 학습 파라미터 비율과 성능의 관계 |
| 2 | 적용 범위 축소 | `target_modules`를 `q_proj`, `v_proj`만 | 파라미터는 줄고 성능은? (원논문 설정) |
| 3 | 학습률 | `5e-5` → `2e-4` | Loss가 빨리 줄어드는가, 불안정해지는가 |
| 4 | 에폭 | `num_train_epochs=1` → `2` | 형식 학습이 더 확실해지는가 |
| 5 | 데이터 양 | `TRAIN_SIZE = 1000` 또는 `19800` | 형식 학습에 몇 건이 필요한가 |
| 6 | **양자화 없이** | `load_in_4bit=False` | 메모리가 얼마나 늘어나는가 (OOM 주의) |

> **⭐ 0번 실험을 가장 먼저 하세요**
>
> 20건으로는 정답률을 판단할 수 없다. **1/20의 95% 신뢰구간은 0.9% ~ 23.6%** 로,
> 무작위(약 3%)와 통계적으로 구분되지 않는다.
> 200건 이상으로 늘려야 비로소 "찍기보다 나은가"를 말할 수 있다.
>
> 👉 **측정이 부정확한 상태에서 개선을 시도하면, 좋아졌는지 나빠졌는지 알 수 없다.**
> 1-1 챕터에서 **교차검증의 표준편차를 함께 보라**고 했던 것과 같은 이유다.

> **⭐ 1번 실험도 권장합니다**
>
> `r`을 4배로 키우면 학습 파라미터도 4배가 된다. **성능도 4배 좋아질까?**
> 대부분 그렇지 않다. **작은 r로도 충분하다**는 것이 LoRA 논문의 핵심 주장이다.
> 직접 확인해 보면 "왜 r=8이 기본값인가"를 이해하게 된다.

### 📚 더 알아보기 — LoRA 이후의 발전

LoRA(2021)는 여전히 표준이지만, 이후 여러 변형이 나왔다. **이름만 알아두자.**

| 기법 | 아이디어 |
|---|---|
| **DoRA** | 가중치를 크기와 방향으로 분해해 각각 학습 |
| **LoRA+** | B와 A에 서로 다른 학습률을 적용 |
| **rsLoRA** | rank가 커질 때의 스케일링을 개선 |
| **AdaLoRA** | 층마다 중요도를 판단해 rank를 다르게 배분 |

> 💡 전부 **"어디에 얼마나 파라미터를 배분할 것인가"** 를 다듬은 것이다.
> **핵심 아이디어(저차원 분해)는 그대로**이므로, LoRA를 이해했다면 나머지는 응용이다.

### 자기주도 실습 안내

이제 `실습_5-1_PEFT_파라미터_효율적_튜닝.ipynb`를 열고,
오늘 배운 개념을 TODO 코드로 직접 구현해 보자.

---

## ➡️ 다음 시간 예고 — 5-2. Quantization

오늘 QLoRA에서 **"모델을 4-bit로 압축한다"** 는 것을 사용했다.
`load_in_4bit=True` 한 줄로 처리했지만, 그 안에서는 많은 일이 일어나고 있다.

**다음 시간에는 그 '압축'의 원리를 파헤친다.**

| 다음 시간에 배울 것 | 오늘과의 연결 |
|---|---|
| **NF4** — 정규분포에 최적화된 4-bit 형식 | 오늘 `load_in_4bit=True`가 쓴 바로 그것 |
| **Double Quantization** | 양자화 상수까지 한 번 더 압축 |
| **PTQ vs QAT** | 언제 양자화하는가의 차이 |
| **추론 최적화** | 오늘은 *학습*용, 다음은 *추론*용 양자화 |

<br>

| 구분 | 5-1 (오늘) | 5-2 (다음 시간) |
|------|:---:|:---:|
| 목적 | **학습** 효율화 | **추론** 효율화 |
| 양자화의 역할 | 베이스 모델을 얼려두는 수단 | **주인공** |
| 결과물 | 도메인 특화 모델 | 가볍고 빠른 배포용 모델 |

<br>

> 💡 **오늘 만든 모델을 실제로 서비스하려면** 다음 시간의 내용이 필요하다.
>
> 학습(5-1)과 배포(5-2)는 한 세트다.


---

## 📌 [참고] 이 기술이 실무에서 쓰이는 방식

오늘 배운 QLoRA는 **개인이나 소규모 팀이 AI 모델을 소유할 수 있게 만든 기술**이다.
실제로 어떤 식으로 활용되는지 몇 가지만 살펴보자.

### 활용 사례

| 사례 | 어떻게 쓰이는가 |
|------|----------------|
| **도메인 특화 어시스턴트** | 법률·의료·금융 문서로 QLoRA 학습 → 전문 용어와 형식을 익힌 모델 |
| **말투·페르소나 학습** | 브랜드 톤앤매너를 학습시켜 일관된 고객 응대 |
| **구조화 출력 강제** | 항상 정해진 JSON 형식으로 답하도록 학습 (2-2의 구조화 출력을 모델 수준에서) |
| **소형 모델 성능 끌어올리기** | 작은 모델을 특정 작업에 특화시켜 큰 모델 수준의 성능 확보 |

### Adapter 교체 전략

LoRA adapter는 **수십 MB**에 불과하다. 이 특성이 실무에서 큰 장점이 된다.

```
   베이스 모델 (수 GB, 서버에 1개만)
        ├── adapter_고객A  (수십 MB)  →  고객A 전용 응대
        ├── adapter_고객B  (수십 MB)  →  고객B 전용 응대
        └── adapter_고객C  (수십 MB)  →  고객C 전용 응대
```

고객사마다 모델을 통째로 복사할 필요 없이 **어댑터만 갈아 끼우면** 된다.
서버 비용과 배포 복잡도가 크게 줄어든다.

### RAG와 함께 쓰기

4-1에서 배운 RAG와 조합하면 서로의 약점을 보완할 수 있다.

```
   Fine-tuning  →  "우리 회사 말투와 답변 형식"을 익힌다
        +
   RAG          →  "최신 정책과 사실 정보"를 실시간으로 공급
        ↓
   말투도 맞고, 사실도 정확한 어시스턴트
```

> 💡 실무에서 가장 많이 쓰이는 조합이다.
> **Fine-tuning은 형식을, RAG는 내용을** 담당한다고 기억하자.

### 핵심 메시지

> **"거대 기업만 AI 모델을 만들 수 있는 시대는 끝났다."**
>
> 오픈 웨이트 모델 + 양자화 + LoRA를 조합하면,
> **16GB GPU 한 장으로도** 실용적인 도메인 특화 모델을 만들 수 있다.
> 오늘 그것을 직접 확인했다.


---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.
